# Experiment Template

**Only edit Cell 1 (CONFIG).** Everything else is boilerplate handled by `src/pipeline.py`.

Workflow:
1. Copy this notebook, rename it (e.g. `18_my_experiment.ipynb`)
2. Change `experiment_name` and adjust `encode_pairs`, `drop_cols`, and model params in CONFIG
3. Run all cells
4. View all experiment runs: `mlflow ui` in the project root → open http://localhost:5000

In [ ]:
# ── EDIT ONLY THIS CELL ────────────────────────────────────────────────────
CONFIG = {
    "experiment_name": "v18",  # Change per experiment — used as MLflow run label

    "data": {
        "train_path": "../../data/raw/train.csv",
        "test_path":  "../../data/raw/test.csv",
    },

    "features": {
        # OOF target encodings — add/remove tuples to test new group price signals
        # Format: (group_column, new_feature_name, min_group_count)
        "encode_pairs": [
            ("town",           "town_median_price",           1),
            ("postal_sector",  "postal_sector_median_price",  1),
            ("flat_model",     "flat_model_median_price",     1),
            ("town_flat_type", "town_flat_type_median_price", 3),
            ("Tranc_Year",     "year_median_price",           1),
        ],
        # Extra columns to drop from X before training (on top of the always-dropped set)
        "drop_cols": [],
    },

    "models": {
        # Remove a model name from 'active' to exclude it from the stack
        "active": ["xgb", "lgbm", "et", "catboost"],

        "xgb": {
            "n_estimators": 2000, "max_depth": 7, "learning_rate": 0.05,
            "subsample": 0.9, "colsample_bytree": 0.4, "min_child_weight": 7,
            "reg_alpha": 0.01, "reg_lambda": 1.5,
            "random_state": 42, "n_jobs": -1, "verbosity": 0, "tree_method": "hist",
        },
        "lgbm": {
            "n_estimators": 1000, "max_depth": 12, "num_leaves": 300,
            "learning_rate": 0.03, "subsample": 0.8, "colsample_bytree": 0.5,
            "min_child_samples": 20, "reg_alpha": 0, "reg_lambda": 0.5,
            "random_state": 42, "n_jobs": -1, "verbosity": -1,
        },
        "et": {
            "n_estimators": 300, "min_samples_split": 10, "min_samples_leaf": 1,
            "max_features": 0.8, "max_depth": 20,
            "random_state": 42, "n_jobs": -1,
        },
        "catboost": {
            "iterations": 1000, "depth": 7, "learning_rate": 0.08,
            "l2_leaf_reg": 1, "colsample_bylevel": 0.7, "min_child_samples": 5,
            "loss_function": "RMSE", "random_seed": 42, "verbose": 0,
        },
        "ridge_alphas": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    },

    "cv": {"n_splits": 5, "random_state": 42},

    "output": {
        # OOF-averaged submission — always generated (honest estimate, each model saw 80% data)
        "submission_path": "../../outputs/submissions/sub_v18.csv",

        # 100% retrain submission — uncomment to enable (~30 min extra, typically $50-100 better on Kaggle)
        # Same Ridge weights as OOF step, only the base models retrain on all 150K rows
        # "fulldata_submission_path": "../../outputs/submissions/sub_v18_fulldata.csv",

        "sample_path": "../../outputs/submissions/sample_sub_reg.csv",
    },
}

In [ ]:
import sys
sys.path.insert(0, "../../")

from src.pipeline import run_pipeline

results = run_pipeline(CONFIG)

In [ ]:
print(f'OOF RMSE:      ${results["oof_rmse"]:,.0f}')
print(f'Ridge weights: {results["weights"]}')
print(f'Best alpha:    {results["best_alpha"]}')
print()
print('Per-model mean OOF RMSE:')
import numpy as np
for m, scores in results['fold_scores'].items():
    print(f'  {m:<12} ${np.mean(scores):>10,.0f}')
print()
print('To view all experiments in MLflow UI:')
print('  cd <project_root>')
print('  mlflow ui --backend-store-uri sqlite:///mlflow.db')
print('  Open http://localhost:5000')